# Two-Occupation Model (v2)

This notebook solves the two-occupation model in two variants:

- discrete experience states
- continuous experience states

Then it compares the solved policy and value functions on aligned state points.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import dcegm

In [ ]:
jax.config.update("jax_enable_x64", True)

params = {
    "interest_rate": 0.02,
    "max_wealth": 50,
    "wage_constant": 3.0,
    "wage_exp_green": 0.5,
    "wage_exp_red": 0.8,
    "income_shock_std": 1.0,
    "income_shock_mean": 0.0,
    "taste_shock_scale": 1.0,
    "discount_factor": 0.95,
    "rho": 0.9,
    "delta": 1.5,
    "beta_green": 0.2,
    "beta_red": 0.1,
}

model_specs = {
    "choices": [0, 1, 2],
}

In [ ]:
def flow_util(consumption, choice, params):
    rho = params["rho"]
    beta_green = params["beta_green"]
    beta_red = params["beta_red"]
    disutility = beta_red * (choice == 0) + beta_green * (choice == 1)
    return consumption ** (1 - rho) / (1 - rho) - disutility


def marginal_utility(consumption, params):
    rho = params["rho"]
    return consumption ** (-rho)


def inverse_marginal_utility(marginal_utility, params):
    rho = params["rho"]
    return marginal_utility ** (-1 / rho)


utility_functions = {
    "utility": flow_util,
    "inverse_marginal_utility": inverse_marginal_utility,
    "marginal_utility": marginal_utility,
}


def final_period_utility(wealth, choice, params):
    return flow_util(wealth, choice, params)


def marginal_final(wealth, choice, params):
    return marginal_utility(wealth, params)


utility_functions_final_period = {
    "utility": final_period_utility,
    "marginal_utility": marginal_final,
}


def state_specific_choice_set(period, lagged_choice, model_specs):
    if lagged_choice == 2:
        return [2]
    if period == 4:
        return [2]
    return model_specs["choices"]

In [ ]:
def next_period_deterministic_state_discrete(
    period, choice, lagged_choice, exp_green, exp_red
):
    return {
        "period": period + 1,
        "exp_green": exp_green + (choice == 1),
        "exp_red": exp_red + (choice == 0),
        "lagged_choice": choice,
    }


def sparsity_condition(period, lagged_choice, exp_green, exp_red):
    if (exp_green + exp_red) > period:
        return False
    if (exp_green > 0) & (lagged_choice == 0) & (period == 1):
        return False
    if (exp_red > 0) & (lagged_choice == 1) & (period == 1):
        return False
    if ((exp_red + exp_green) == 0) & (lagged_choice != 2):
        return False
    return True


def budget_constraint_discrete_exp(
    lagged_choice,
    exp_green,
    exp_red,
    asset_end_of_previous_period,
    income_shock_previous_period,
    params,
):
    interest_factor = 1 + params["interest_rate"]
    wage = (
        params["wage_constant"]
        + params["wage_exp_green"] * exp_green * (lagged_choice == 1)
        + params["wage_exp_red"] * exp_red * (lagged_choice == 0)
    )
    resource = (
        interest_factor * asset_end_of_previous_period
        + (wage + income_shock_previous_period) * (lagged_choice != 2)
        + (wage + income_shock_previous_period) * 0.5 * (lagged_choice == 2)
    )
    return jnp.maximum(resource, 0.5)


model_config_discrete = {
    "n_periods": 5,
    "choices": [0, 1, 2],
    "continuous_states": {
        "assets_end_of_period": jnp.linspace(0, 50, 100),
        "assets_begin_of_period": jnp.linspace(0, 50, 100),
    },
    "deterministic_states": {
        "exp_green": jnp.arange(0, 7, dtype=int),
        "exp_red": jnp.arange(0, 7, dtype=int),
    },
    "n_quad_points": 5,
    "upper_envelope": {"method": "druedahl_jorgensen"},
}

state_space_functions_discrete = {
    "state_specific_choice_set": state_specific_choice_set,
    "next_period_deterministic_state": next_period_deterministic_state_discrete,
    "sparsity_condition": sparsity_condition,
}

In [ ]:
def next_period_deterministic_state_cont(period, choice, lagged_choice):
    return {
        "period": period + 1,
        "lagged_choice": choice,
    }


def next_period_continuous_state(lagged_choice, period, exp_green, exp_red):
    exp_red_years = period * exp_red
    exp_green_years = period * exp_green
    add_red = lagged_choice == 0
    add_green = lagged_choice == 1

    period_scale = period.clip(min=1)
    exp_red_lag_years = exp_red_years * (period_scale - 1).clip(min=0)
    exp_red_next = (exp_red_lag_years + add_red) / period_scale

    exp_green_lag_years = exp_green_years * (period_scale - 1).clip(min=0)
    exp_green_next = (exp_green_lag_years + add_green) / period_scale

    return {
        "exp_red": exp_red_next,
        "exp_green": exp_green_next,
    }


def budget_constraint_cont_exp(
    period,
    lagged_choice,
    exp_green,
    exp_red,
    asset_end_of_previous_period,
    income_shock_previous_period,
    params,
):
    exp_green_years = exp_green * period
    exp_red_years = exp_red * period
    interest_factor = 1 + params["interest_rate"]
    wage = (
        params["wage_constant"]
        + params["wage_exp_green"] * exp_green_years * (lagged_choice == 1)
        + params["wage_exp_red"] * exp_red_years * (lagged_choice == 0)
    )
    resource = (
        interest_factor * asset_end_of_previous_period
        + (wage + income_shock_previous_period) * (lagged_choice != 2)
        + (wage + income_shock_previous_period) * 0.5 * (lagged_choice == 2)
    )
    return jnp.maximum(resource, 0.5)


model_config_cont = {
    "n_periods": 5,
    "choices": [0, 1, 2],
    "continuous_states": {
        "assets_end_of_period": jnp.linspace(0, 50, 100),
        "assets_begin_of_period": jnp.linspace(0, 50, 100),
        "exp_green": jnp.linspace(0, 1, 5, dtype=float),
        "exp_red": jnp.linspace(0, 1, 5, dtype=float),
    },
    "n_quad_points": 5,
    "upper_envelope": {"method": "druedahl_jorgensen"},
}

state_space_functions_cont = {
    "state_specific_choice_set": state_specific_choice_set,
    "next_period_deterministic_state": next_period_deterministic_state_cont,
    "next_period_continuous_state": next_period_continuous_state,
}

In [ ]:
model_discrete = dcegm.setup_model(
    model_config=model_config_discrete,
    model_specs=model_specs,
    utility_functions=utility_functions,
    utility_functions_final_period=utility_functions_final_period,
    state_space_functions=state_space_functions_discrete,
    stochastic_states_transitions={},
    budget_constraint=budget_constraint_discrete_exp,
)

model_cont = dcegm.setup_model(
    model_config=model_config_cont,
    model_specs=model_specs,
    utility_functions=utility_functions,
    utility_functions_final_period=utility_functions_final_period,
    state_space_functions=state_space_functions_cont,
    stochastic_states_transitions={},
    budget_constraint=budget_constraint_cont_exp,
)

sol_discrete = model_discrete.solve(params)
sol_cont = model_cont.solve(params)

In [ ]:
assets = jnp.linspace(0.5, 20.0, 80)
period = jnp.full_like(assets, 3, dtype=int)
lagged_choice = jnp.full_like(assets, 0, dtype=int)
choice = jnp.full_like(assets, 0, dtype=int)

states_discrete = {
    "period": period,
    "lagged_choice": lagged_choice,
    "exp_green": jnp.full_like(assets, 1, dtype=int),
    "exp_red": jnp.full_like(assets, 1, dtype=int),
    "assets_begin_of_period": assets,
}

states_cont = {
    "period": period,
    "lagged_choice": lagged_choice,
    "exp_green": jnp.full_like(assets, 1.0 / 3.0),
    "exp_red": jnp.full_like(assets, 1.0 / 3.0),
    "assets_begin_of_period": assets,
}

policy_discrete = sol_discrete.policy_for_states_and_choices(states_discrete, choice)
value_discrete = sol_discrete.value_for_states_and_choices(states_discrete, choice)

policy_cont = sol_cont.policy_for_states_and_choices(states_cont, choice)
value_cont = sol_cont.value_for_states_and_choices(states_cont, choice)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(assets, policy_discrete, label="discrete exp")
axes[0].plot(assets, policy_cont, label="continuous exp", linestyle="--")
axes[0].set_title("Policy Function (choice=0)")
axes[0].set_xlabel("Assets at beginning of period")
axes[0].set_ylabel("Consumption")
axes[0].legend()

axes[1].plot(assets, value_discrete, label="discrete exp")
axes[1].plot(assets, value_cont, label="continuous exp", linestyle="--")
axes[1].set_title("Value Function (choice=0)")
axes[1].set_xlabel("Assets at beginning of period")
axes[1].set_ylabel("Value")
axes[1].legend()

plt.tight_layout()
plt.show()